# Lab 09 · Multi-Source Knowledge Routing (Notebook Walkthrough)

**Concept.** A real assistant rarely searches a single corpus. This lab adds a **second, separate index** on a completely different topic — *the future of generative AI* — next to the deep-excavation engineering corpus the earlier labs built. With two indexes registered as Azure AI Search **knowledge sources**, the assistant decides, per question, **which index (or indexes) to search**.

There are **two routing layers**, depending on the retrieval mode:
- **Agentic retrieval** (`auto` chat, the deployed assistant's default) **delegates source selection to the LLM query planner** — the app hands *every* source to the knowledge base and the planner picks per subquery (`routing_mode: planner_delegated`).
- **Direct search** (`hybrid` / `vector` / `full_text`) has no planner, so the app's **keyword router** selects indexes (`keyword_routed`, `cross_source_intent`, `broad_auto`, `primary_default`). The route preview and hybrid demo below exercise this path so you can *see* the decision.

You will see three routing outcomes in **direct search**:
1. an **AI-trends** question routed to the AI-trends index only,

2. an **excavation** question routed to the engineering index only,> Prerequisite: the second source must be registered before you start. Set `AZURE_SEARCH_EXTRA_SOURCES_JSON` in your `.env` (see the lab doc), then restart the app. The app auto-creates the `ai-trends-index` on first ingest — you do not have to create it by hand.

3. a **compare** question fanned out across **both** indexes.

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': '<home>\\rag-on-azure-lab',
 'env_values_loaded': 0,
 'search_endpoint': 'https://your-search-service.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Confirm the second knowledge source is registered

`AZURE_SEARCH_EXTRA_SOURCES_JSON` is parsed at startup into `settings.azure_search_extra_sources`. If the list below is empty, set the variable in `.env` and restart the kernel before continuing.

In [2]:
from backend.core.config import settings
extra = [(s.knowledge_source_name, s.index_name) for s in settings.azure_search_extra_sources]
print('Extra knowledge sources:', extra or 'NONE — set AZURE_SEARCH_EXTRA_SOURCES_JSON and restart')
for s in settings.azure_search_extra_sources:
    print(' route_keywords:', s.route_keywords)
    print(' assignment_keywords:', s.assignment_keywords)

Extra knowledge sources: [('ai-trends-source', 'ai-trends-index')]
 route_keywords: ('artificial intelligence', 'machine learning', 'trends', 'future', 'innovation', 'forecast', 'generative', 'models', 'adoption')
 assignment_keywords: ('future', 'trends', 'innovation', 'forecast', 'artificial intelligence', 'machine learning', 'generative')


## Step 3 — Ingest both corpora

The engineering corpus already exists from the earlier labs; the AI-trends PDF is ingested into the **separate** `ai-trends-index`. Ingest-time routing reads the document's name and section headings and matches them against each source's `assignment_keywords` — the filename tokens `future` and `trends` send this document to the AI-trends source. Both runs reuse cached results when available.

In [3]:
ai_job = lab.ingest(pdf_path='data/ai-future-trends.pdf', skill_profile='genai_enrichment', reuse=True)
try:
    exc_job = lab.ingest(skill_profile='genai_enrichment', reuse=True)
except FileNotFoundError:
    # The engineering corpus was already ingested by earlier labs; reuse it without the source file.
    exc_job = lab.find_existing_job(skill_profile_id='genai_enrichment',
                                    file_name='Deep Excavation Design and Construction.pdf')

ai_diag = ai_job.publish_status.diagnostics or {}
exc_diag = (exc_job.publish_status.diagnostics or {}) if exc_job else {}
print('Excavation doc ->', exc_diag.get('index_name'))
print('AI-trends doc  ->', ai_diag.get('index_name'),
      '| assignment_mode:', ai_diag.get('assignment_mode'),
      '| matched:', ai_diag.get('assignment_matches'))
print('All publishable indexes:', ai_diag.get('index_names'))

Reusing existing 'genai_enrichment' ingestion (doc_id=8db477c5, 108 chunks). Pass reuse=False to force a fresh run.
Excavation doc -> ai-search-lab-index
AI-trends doc  -> ai-trends-index | assignment_mode: keyword_assigned | matched: ['future', 'trends']
All publishable indexes: ['ai-search-lab-index', 'ai-trends-index']


## Step 4 — Preview routing *before* searching (direct search)

`route_preview` runs the same keyword router the **direct-search** retrieve path uses, but stops before issuing any request — so it is instant and free. Watch the `routing_mode` and the indexes it selects for each question.

> This previews the **direct-search** keyword router. **Agentic** retrieval (Steps 6–7) skips this router and delegates source selection to the LLM planner instead (`routing_mode: planner_delegated`).

In [4]:
Q_AI = 'What does the report forecast about generative AI adoption trends and the future of the technology?'
Q_EXC = 'What groundwater control measures are recommended to support a deep excavation?'
Q_COMPARE = 'Compare how risk and forecasting are treated across the two indexes.'

import pandas as pd

rows = []
for label, q in [('AI trends', Q_AI), ('Excavation', Q_EXC), ('Compare', Q_COMPARE)]:
    rp = lab.route_preview(q)
    rows.append({
        'question': label,
        'routing_mode': rp['routing_mode'],
        'selected_indexes': ', '.join(rp['selected_search_indexes']),
        'matched_terms': '; '.join(
            f"{idx}:{terms}" for idx, terms in rp['matched_terms_by_index'].items() if terms
        ),
    })
pd.DataFrame(rows)

,question,routing_mode,selected_indexes,matched_terms
0,AI trends,keyword_routed,ai-trends-index,"ai-trends-index:['trends', 'future', 'forecast..."
1,Excavation,keyword_routed,ai-search-lab-index,ai-search-lab-index:['excavation']
2,Compare,cross_source_intent,"ai-search-lab-index, ai-trends-index",


### Read the routing table (direct search)

- **AI trends** → `keyword_routed` to the **AI-trends index** because the question matched that source's `route_keywords` (trends, future, forecast, generative…).
- **Excavation** → `keyword_routed` to the **primary engineering index** — the term *excavation* matches the corpus published there.
- **Compare** → `cross_source_intent`: compare/across language tells the router to query **both** indexes so the answer can draw from each.

> Reminder: these modes are how the *direct-search* path scopes a query. Agentic retrieval doesn't use them — it passes all sources to the planner.

## Step 5 — Prove it: hybrid search shows which index served each hit

No `doc_ids` are pinned, so the **router** (not a manual selection) decides the scope. Each hit carries the index it came from.

In [5]:
hits, diag = lab.multi_source_search(Q_AI, retrieval_mode='hybrid', top=5)
print('routing_mode:', diag.get('routing_mode'), '| indexes:', diag.get('selected_search_indexes'))
pd.DataFrame(hits)

routing_mode: keyword_routed | indexes: ['ai-trends-index']


,index,knowledge_source,score,source_doc,snippet
0,ai-trends-index,ai-trends-source,0.0156,ai-future-trends.pdf,McKinsey Explainers (Aug. 2023) provides an ea...
1,ai-trends-index,ai-trends-source,0.0331,ai-future-trends.pdf,McKinsey Explainers (Aug. 2023) provides an ea...
2,ai-trends-index,ai-trends-source,0.0141,ai-future-trends.pdf,A new Mckinsey survey shows that the vast majo...
3,ai-trends-index,ai-trends-source,0.0143,ai-future-trends.pdf,McKinsey Explainers (Aug. 2023) provides an ea...
4,ai-trends-index,ai-trends-source,0.0159,ai-future-trends.pdf,McKinsey Explainers (Aug. 2023) provides an ea...


In [6]:
hits, diag = lab.multi_source_search(Q_EXC, retrieval_mode='hybrid', top=5)
print('routing_mode:', diag.get('routing_mode'), '| indexes:', diag.get('selected_search_indexes'))
pd.DataFrame(hits)

routing_mode: keyword_routed | indexes: ['ai-search-lab-index']


,index,knowledge_source,score,source_doc,snippet
0,ai-search-lab-index,ai-search-lab-source,0.0147,Deep Excavation Design and Construction.pdf,"GEO Publication No. 1/2023, **Deep Excavation ..."
1,ai-search-lab-index,ai-search-lab-source,0.0145,Deep Excavation Design and Construction.pdf,"GEO Publication No. 1/2023, **Deep Excavation ..."
2,ai-search-lab-index,ai-search-lab-source,0.0139,Deep Excavation Design and Construction.pdf,"GEO Publication No. 1/2023, **Deep Excavation ..."
3,ai-search-lab-index,ai-search-lab-source,0.0101,Deep Excavation Design and Construction.pdf,"GEO Publication No. 1/2023, **Deep Excavation ..."
4,ai-search-lab-index,ai-search-lab-source,0.0186,Deep Excavation Design and Construction.pdf,"For excavation below groundwater level, it is ..."


Each question's hits come **only** from the relevant index — the AI question never pulls excavation chunks, and vice versa. That isolation is what keeps grounded answers on-topic in a multi-corpus assistant.

## Step 6 — A grounded answer where the planner picks the source

`ask_corpus` runs a full chat turn in `auto` corpus mode — **agentic** retrieval, exactly like the deployed assistant. Here the app hands **every** ready source to the knowledge base and the **LLM query planner** decides which to query (`routing_mode: planner_delegated`). The AI question is answered from the AI-trends index because the planner's subqueries match there.

In [7]:
resp = lab.ask_corpus(Q_AI, retrieval_mode='agentic')
lab.show_answer(resp, max_citations=4)

[retrieval_mode=agentic | scoring_profile=default | citations=8]

The report forecasts that generative AI adoption is already moving very quickly and will continue to evolve rapidly. [1] It says many workers across industries and regions have already tried gen AI tools, describing this as “pretty rapid adoption” in less than one year. [2] It also suggests the technology’s capabilities will keep improving fast: for most technical capabilities shown, gen AI is expected to perform at a median level of human performance by the end of this decade, and in some cases to compete with the top 25 percent of people before 2040. [3] The report says this could happen much sooner than experts previously thought. [3] Looking ahead, the report expects gen AI to create value across many uses, with the biggest near-term gains coming from specific industry- and function-focused applications rather than general-purpose ones. [4] It also says nearly all industries should see significant gains from marketin

## Step 7 — The cross-source question, handled by the planner

In agentic mode the planner receives **both** sources and decomposes the compare question into subqueries that hit each corpus — no compare-keyword heuristic required. It then grounds the synthesized answer in whichever retrieved chunks are most relevant, so the citations may lean toward one corpus even though both were searched. (Contrast with Step 4, where the *direct-search* router needed explicit `cross_source_intent` language to fan out.)

In [8]:
resp = lab.ask_corpus(Q_COMPARE, retrieval_mode='agentic')
lab.show_answer(resp, max_citations=6)

[retrieval_mode=agentic | scoring_profile=default | citations=6]

I found only one relevant index-like source in the retrieved material, which treats both topics as follows: - **Risk:** The document explicitly says gen AI “doesn’t come without potential risks,” including biased, factually wrong, or illegally scraped content, and it warns organizations about reputational and legal risks; it recommends keeping a human in the loop to mitigate those risks. [1]
- **Forecasting:** The same document frames gen AI’s future in terms of what it will be good at and how quickly, which jobs and industries it will affect, what activities create the most value, how workers feel, and what safeguards are needed. It also presents a charted timeline of major LLM developments after ChatGPT’s launch. [2][3] If you meant **two specific indexes**, I only have evidence for one in the retrieved content, so I can’t make a true side-by-side comparison from the available material. [2][1]

Citations:
  [1] Organiz

## Takeaways

- A knowledge base can reference **multiple knowledge sources**, each backed by its **own index**.
- **Ingest-time** routing (`assignment_keywords`) decides which index a document lands in. **Query-time** source selection depends on the mode:
  - **Agentic** retrieval → the **LLM planner** selects sources (`planner_delegated`); the app passes all sources and the planner picks per subquery. Keep each source's `description` accurate so the planner can tell corpora apart.
  - **Direct search** → the app's **keyword router** selects indexes: `keyword_routed` (a source's hints matched), `cross_source_intent` (compare/across language → all sources), `broad_auto` (no hint matched but few enough sources to fan out), `primary_default` (fall back to the main index).
- Tune `route_keywords` per source and `AZURE_SEARCH_AUTO_BROADCAST_LIMIT` to balance precision against recall **for direct search**; the agentic planner is unaffected by these.

Next: the **comparison notebook** lays every retrieval method side by side.